In [156]:
import scipy.io
import pprint
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

In [157]:
NORMALIZATION_VALUE = 300
DEFAULT_NUMBER_OF_VALUES_IN_WINDOW = 10

TRAINING_DATA_PERCENTAGE = 0.85
DEFAULT_BATCH_SIZE = 8

In [158]:
def createDataWithWindows(d, windowSize):
    windowedData = []

    for i in range(len(d)):
        windowValues = []

        for j in range(windowSize):
            if(i - windowSize + j < 0):
                windowValues.append(0.0)
            else:
                windowValues.append(d[i - windowSize + j])

        windowedData.append([windowValues, d[i]])

    return windowedData

def saveData(d, windowSize):
    np.save('data/data_' + str(windowSize) + '.npy', np.array(d, dtype = object))

def loadData(windowSize):
    data = np.load('data/data_' + str(windowSize) + '.npy', allow_pickle = True).tolist()
    return data

def createWindowedDataFiles(d, min, max):
    for i in range(min, max):
        temp = createDataWithWindows(d, i)
        saveData(temp, i)

def printData(d):
    for item in d:
        print(item)

In [159]:
# Load data
mat = scipy.io.loadmat('Xtrain.mat')

data = mat['Xtrain']
data = data[:, 0] 
data = data / NORMALIZATION_VALUE

createWindowedDataFiles(data, 10, 20)

# Example usage
data = createDataWithWindows(data, DEFAULT_NUMBER_OF_VALUES_IN_WINDOW)
saveData(data, DEFAULT_NUMBER_OF_VALUES_IN_WINDOW)
data = loadData(DEFAULT_NUMBER_OF_VALUES_IN_WINDOW)

printData(data)

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], np.float64(0.2866666666666667)]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(0.2866666666666667)], np.float64(0.47)]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(0.2866666666666667), np.float64(0.47)], np.float64(0.31666666666666665)]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(0.2866666666666667), np.float64(0.47), np.float64(0.31666666666666665)], np.float64(0.13666666666666666)]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(0.2866666666666667), np.float64(0.47), np.float64(0.31666666666666665), np.float64(0.13666666666666666)], np.float64(0.07333333333333333)]
[[0.0, 0.0, 0.0, 0.0, 0.0, np.float64(0.2866666666666667), np.float64(0.47), np.float64(0.31666666666666665), np.float64(0.13666666666666666), np.float64(0.07333333333333333)], np.float64(0.07)]
[[0.0, 0.0, 0.0, 0.0, np.float64(0.2866666666666667), np.float64(0.47), np.float64(0.31666666666666665), np.float64(0.13666666666666666), np.float64(0.073333333

In [160]:
class WindowedDataset(Dataset):
    def __init__(self, data):
        self.X = torch.tensor([item[0] for item in data], dtype=torch.float32)
        self.y = torch.tensor([item[1] for item in data], dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    

trainDataSet = WindowedDataset(data)

trainSetSize = int(TRAINING_DATA_PERCENTAGE * len(trainDataSet))
validSetSize = len(trainDataSet) - trainSetSize
trainDataSet, validDataSet = torch.utils.data.random_split(trainDataSet, [trainSetSize, validSetSize])

trainLoader = DataLoader(trainDataSet, batch_size = DEFAULT_BATCH_SIZE, shuffle = True)
validLoader = DataLoader(validDataSet, batch_size = DEFAULT_BATCH_SIZE, shuffle = True)